# SNN full study on Kaggle

Before running:

- `git push` your repo first (Kaggle clones from GitHub).
- Settings: GPU (T4 x2 / P100), Internet On, Persistence = Variables and Files.
- One mechanism per session (~9h limit); persist `results/` as a Dataset (last cell).
- Edit `REPO_URL` in the next cell.

In [ ]:
# 1. Clone + install
import os, subprocess, sys

REPO_URL = "https://github.com/thisisrick25/snn.git"  # EDIT ME (private repo: use a token URL)
REPO_DIR = "/kaggle/working/snn"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR

# torch/torchvision/numpy ship with Kaggle images; do NOT pin +cu132 (code is device-agnostic via device: auto)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "snntorch", "pyyaml"], check=True)

import torch
print("cwd:", os.getcwd())
print("cuda available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. (Optional) Restore prior results from a Kaggle Dataset input, so --resume can skip finished work.
# Add your saved dataset via 'Add Input' (right panel); it mounts read-only under /kaggle/input/<slug>/.
import glob, os, shutil

dest = "results/full_study/raw"
os.makedirs(dest, exist_ok=True)
restored = 0
for raw_dir in glob.glob("/kaggle/input/*/results/full_study/raw") + glob.glob("/kaggle/input/*/raw"):
    for f in glob.glob(os.path.join(raw_dir, "*.json")):
        shutil.copy(f, dest)
        restored += 1
print(f"restored {restored} raw JSON(s) into {dest}")

In [ ]:
# 3. Run ONE cell of the matrix this session (edit the variables below), then re-run this cell.
# --resume is ON by default: finished (seed, condition) pairs are skipped automatically.
#
# Full 12-run matrix (2 settings x 3 mechanisms core, + 2 settings x 3 controls at threshold):
#   CORE:    (mnist, mlp | cifar10, conv_snn) x (kwta_window | activity_reg | threshold)
#   CONTROL: (mnist, mlp | cifar10, conv_snn) x --sparsity-mode threshold
#            --control (update_norm | activation_dropout | block_freeze)
# Tip: start with mnist/mlp/kwta_window (fastest); give each CIFAR-Conv-SNN mechanism its own session.

DATASET = "mnist"        # mnist | cifar10
ARCH = "mlp"             # mlp | conv_snn
SPARSITY = "kwta_window" # kwta_window | activity_reg | threshold
CONTROL = "none"         # none | update_norm | activation_dropout | block_freeze (controls use SPARSITY=threshold)

cmd = [
    "python", "-m", "src.scripts.run_pilot",
    "--config", "configs/full_study.yaml",
    "--dataset", DATASET, "--arch", ARCH, "--sparsity-mode", SPARSITY,
]
if CONTROL != "none":
    cmd += ["--control", CONTROL]
print("running:", " ".join(cmd))
import subprocess; subprocess.run(cmd, check=True)

In [ ]:
# 3b. (Alternative) Let the skip-aware wrapper walk the whole matrix; it skips completed cells.
# Good for an unattended session, but a single session may not finish CIFAR-Conv-SNN.
# !bash run_full_study.sh

In [ ]:
# 4. Progress check: how many raw JSONs per cell (expected: kwta 36, activity_reg 36, threshold 81, each control 9).
import glob
from collections import Counter

counts = Counter()
for f in glob.glob("results/full_study/raw/*.json"):
    parts = os.path.basename(f).split("_")
    if len(parts) >= 5:
        counts["_".join(parts[1:5])] += 1  # dataset_arch_mechanism_control
for k in sorted(counts):
    print(f"{counts[k]:>3}  {k}")

In [ ]:
# 5. Final analysis (run only after ALL 12 cells are complete).
import subprocess
subprocess.run(["python", "-m", "src.scripts.run_confirmatory", "--config", "configs/full_study.yaml"], check=True)
print(open("results/full_study/metrics/confirmatory.json").read())

In [ ]:
# 6. Save results so the next session can resume.
# Copies results/full_study into /kaggle/working/out (kept when you 'Save Version').
# For cross-session resume, also create/version a Kaggle Dataset from this folder, then
# 'Add Input' it next session so cell 2 can restore it.
import shutil, os
out = "/kaggle/working/out/full_study"
if os.path.isdir(out):
    shutil.rmtree(out)
shutil.copytree("results/full_study", out)
print("saved to", out)

## Notes

- `/kaggle/working` is wiped between sessions: save results as a Dataset (cell 6), re-add it next session, re-run cell 2. `--resume` skips finished work.
- Let Kaggle generate activity_reg fresh; don't import old local `*_activity_reg_*.json`.
- To redo a cell: delete its raw JSONs and re-run cell 3, or append `--no-resume`.